In [5]:
import pandas as pd

df = pd.read_csv("inventory_clean.csv") 

print(df.shape)
print(df.columns.tolist())
print(df.dtypes)
print(df.head())
print(df.isnull().sum())

(73100, 16)
['Sale_Date', 'Store_ID', 'Product_ID', 'Category', 'Region', 'Inventory_Level', 'Units_Sold', 'Units_Ordered', 'Demand_Forecast', 'Price', 'Discount', 'Weather_Condition', 'Holiday/Promotion', 'Competitor_Pricing', 'Seasonality', 'Revenue']
Sale_Date              object
Store_ID               object
Product_ID             object
Category               object
Region                 object
Inventory_Level         int64
Units_Sold              int64
Units_Ordered           int64
Demand_Forecast       float64
Price                 float64
Discount                int64
Weather_Condition      object
Holiday/Promotion       int64
Competitor_Pricing    float64
Seasonality            object
Revenue               float64
dtype: object
    Sale_Date Store_ID Product_ID     Category Region  Inventory_Level  \
0  2022-01-01     S001      P0001    Groceries  North              231   
1  2022-01-01     S001      P0002         Toys  South              204   
2  2022-01-01     S001      P0

In [7]:
print(df['Revenue'].sum())

494971396.22


In [9]:
df['Sale_Date'] = pd.to_datetime(df['Sale_Date'])
print(df['Sale_Date'].dtype)   
print(df['Sale_Date'].min(), df['Sale_Date'].max())  

datetime64[ns]
2022-01-01 00:00:00 2024-01-01 00:00:00


In [21]:
# Overall business numbers

print("Total revenue:", round(df['Revenue'].sum(), 2))
print("Total units sold:", df['Units_Sold'].sum())
print("Avg daily revenue:", round(df.groupby('Sale_Date')['Revenue'].sum().mean(), 2))
print("Date range:", df['Sale_Date'].min(), "to", df['Sale_Date'].max())

Total revenue: 494971396.22
Total units sold: 9975582
Avg daily revenue: 677115.45
Date range: 2022-01-01 00:00:00 to 2024-01-01 00:00:00


In [13]:
# Revenue by category, region, and store

print(df.groupby('Category')['Revenue'].sum().sort_values(ascending=False))
print()
print(df.groupby('Region')['Revenue'].sum().sort_values(ascending=False))
print()
print(df.groupby('Store_ID')['Revenue'].sum().sort_values(ascending=False))

Category
Furniture      1.002308e+08
Groceries      9.994897e+07
Toys           9.872922e+07
Clothing       9.867678e+07
Electronics    9.738559e+07
Name: Revenue, dtype: float64

Region
East     1.247922e+08
South    1.245674e+08
North    1.235437e+08
West     1.220681e+08
Name: Revenue, dtype: float64

Store_ID
S005    1.005036e+08
S003    1.000344e+08
S002    9.895001e+07
S004    9.805714e+07
S001    9.742619e+07
Name: Revenue, dtype: float64


In [15]:
# Revenue trend over time

monthly = df.groupby(df['Sale_Date'].dt.to_period('M'))['Revenue'].sum()
print(monthly)

Sale_Date
2022-01    20919533.56
2022-02    19061034.22
2022-03    21522684.08
2022-04    20314403.45
2022-05    20542563.15
2022-06    20981787.41
2022-07    21293911.26
2022-08    21055282.07
2022-09    20628911.79
2022-10    21147497.08
2022-11    20719138.27
2022-12    20268138.55
2023-01    20723764.21
2023-02    19228947.15
2023-03    20944689.56
2023-04    20197979.03
2023-05    20726725.71
2023-06    20046289.60
2023-07    21306955.24
2023-08    20566571.67
2023-09    19876340.94
2023-10    20912917.19
2023-11    20711676.59
2023-12    20645511.66
2024-01      628142.78
Freq: M, Name: Revenue, dtype: float64


In [17]:
# Stock-out risk

df['near_stockout'] = df['Units_Sold'] >= df['Inventory_Level'] * 0.9

print("Share of rows flagged as near-stockout:", round(df['near_stockout'].mean(), 4))
print()
print(df.groupby('Category')['near_stockout'].mean().sort_values(ascending=False))

Share of rows flagged as near-stockout: 0.1022

Category
Groceries      0.104579
Furniture      0.104565
Clothing       0.103036
Toys           0.099843
Electronics    0.099029
Name: near_stockout, dtype: float64


In [23]:
# Days of supply by category

days_of_supply = (
    df.groupby('Category')
      .apply(lambda g: g['Inventory_Level'].mean() / g['Units_Sold'].mean())
      .sort_values()
)
print(days_of_supply)

Category
Furniture      2.002069
Clothing       2.008971
Toys           2.013083
Groceries      2.014047
Electronics    2.018526
dtype: float64


/var/folders/80/df4bbr5j4n3gr3hgn96x9q7r0000gn/T/ipykernel_34238/892012915.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g['Inventory_Level'].mean() / g['Units_Sold'].mean())


In [25]:
# Forecast accuracy

df['forecast_error'] = df['Demand_Forecast'] - df['Units_Sold']
df['forecast_error_pct'] = df['forecast_error'] / df['Units_Sold'].replace(0, pd.NA)

print(df.groupby('Category')['forecast_error'].mean())
print()
print(df.groupby('Category')['forecast_error_pct'].mean())

Category
Clothing       5.095996
Electronics    5.029166
Furniture      5.091099
Groceries      4.986473
Toys           5.116025
Name: forecast_error, dtype: float64

Category
Clothing       0.165123
Electronics    0.151295
Furniture      0.159357
Groceries      0.168451
Toys           0.164627
Name: forecast_error_pct, dtype: object


In [27]:
# Does discount or promotion move units sold?

print(df.groupby('Discount')['Units_Sold'].mean())
print()
print(df.groupby('Holiday/Promotion')['Units_Sold'].mean())

Discount
0     135.694585
5     136.567405
10    136.769851
15    136.655293
20    136.640775
Name: Units_Sold, dtype: float64

Holiday/Promotion
0    136.505375
1    136.423926
Name: Units_Sold, dtype: float64


In [29]:
# If forecast overestimates, how many "phantom" units does that represent?
total_forecast = df['Demand_Forecast'].sum()
total_actual = df['Units_Sold'].sum()
print("Total forecasted units:", round(total_forecast))
print("Total actual units sold:", total_actual)
print("Overforecast units:", round(total_forecast - total_actual))

# Rough dollar value of the overforecast, using average price
avg_price = df['Price'].mean()
print("Approx. dollar value of overforecast:", round((total_forecast - total_actual) * avg_price, 2))

Total forecasted units: 10345750
Total actual units sold: 9975582
Overforecast units: 370168
Approx. dollar value of overforecast: 20409248.32


In [31]:
print("Overforecast as % of total revenue:", round(20409248.32 / df['Revenue'].sum() * 100, 2), "%")
print("Overforecast as % of total units sold:", round(370168 / df['Units_Sold'].sum() * 100, 2), "%")

Overforecast as % of total revenue: 4.12 %
Overforecast as % of total units sold: 3.71 %


In [33]:
cat_overforecast = (
    df.groupby('Category')
      .apply(lambda g: (g['Demand_Forecast'].sum() - g['Units_Sold'].sum()))
)
cat_avg_price = df.groupby('Category')['Price'].mean()

cat_value = (cat_overforecast * cat_avg_price).round(2)
print(cat_value.sort_values(ascending=False))
print()
print("Total:", cat_value.sum())

Category
Furniture      4129033.69
Toys           4122695.21
Clothing       4090920.18
Electronics    4039263.16
Groceries      4026914.74
dtype: float64

Total: 20408826.98


/var/folders/80/df4bbr5j4n3gr3hgn96x9q7r0000gn/T/ipykernel_34238/637816545.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: (g['Demand_Forecast'].sum() - g['Units_Sold'].sum()))


In [35]:
monthly_bias = (
    df[df['Sale_Date'] < '2024-01-01']  # exclude the partial last month
    .groupby(df['Sale_Date'].dt.to_period('M'))
    .apply(lambda g: g['Demand_Forecast'].sum() - g['Units_Sold'].sum())
)
print(monthly_bias)

Sale_Date
2022-01    15751.14
2022-02    14524.36
2022-03    16037.92
2022-04    15167.25
2022-05    15022.30
2022-06    15083.12
2022-07    15661.39
2022-08    15532.18
2022-09    15576.17
2022-10    15363.41
2022-11    15067.89
2022-12    15715.84
2023-01    15572.70
2023-02    15104.38
2023-03    15657.48
2023-04    14370.87
2023-05    14844.00
2023-06    15588.24
2023-07    16720.64
2023-08    16321.52
2023-09    14644.48
2023-10    15472.01
2023-11    15923.62
2023-12    14959.75
Freq: M, dtype: float64
